# Long-read DeepVariant **global** PCA (Hail / Terra)

Build whole-cohort ancestry PCs from autosomal, biallelic, LD-pruned DeepVariant
SNVs using Hail on Terra. Within-population PCs live in
`tractor_05b_pca_within_population.ipynb` (run after covariates have complete
ancestry labels).

This notebook is **genotype-only**: it does not read covariates. It writes
versioned `lr_PC*` outputs and a QC MatrixTable checkpoint for the follow-up.

## Spark / Hail cluster (Terra)

Workload class: ~12k joint-callset samples × 22 autosomal VCF shards → QC MT
checkpoint → LD prune → global PCA.
`ld_prune` and `hwe_normalized_pca` are the memory bottlenecks; VCF import is
the I/O / cost bottleneck.

### Recommended first full run

| Role | Machine type | Count | Notes |
| --- | --- | --- | --- |
| Master / driver | `n1-highmem-8` or `n2-highmem-8` | 1 | 8 vCPU / ~52 GB; keep headroom for exports and Spark UI |
| Workers | `n1-highmem-8` or `n2-highmem-8` | **32–50** | Prefer highmem (≥~6.5 GB/core). Standard nodes often OOM on LD prune / PCA |
| Worker disk | SSD | **200–500 GB** each | Shuffle spill + VCF decompress; 100 GB is usually too tight |
| Preemptibles | optional | ≤ half of workers | Fine for import + `variant_qc` / `sample_qc`; risky for long LD-prune / PCA stages |

Start around **40 × `n1-highmem-8`**. Scale toward 50–80 if import or prune is slow; do not “fix” OOMs by adding standard (low-memory) nodes.

### If you already have `qc_for_pca.mt`

Reuse the checkpoint and run PCA-only on **16–32** highmem workers. That is cheaper
and avoids re-importing the 22 chrom shards.

### Spark knobs if you hit memory pressure

- Prefer fewer, fatter executors on highmem nodes, e.g. `spark.executor.cores=4`
  so each executor gets more RAM (set in `hl.init` if the environment allows).
- Raise `LD_MEMORY_PER_CORE` in the config cell (Hail expects an int storage capacity in GiB; start at `1`; try `2` on OOM).
- Checkpoint after QC (`CHECKPOINT_MT`) before prune/PCA so retries are cheap.
- Tighten AF / call-rate filters only if prune is still too wide after that.


### Resume / retry

Re-running the notebook skips completed stages when outputs already exist under
the same `PCA_RUN_LABEL` / `OUT_DIR`:

| Stage | Artifact | Skip when present |
| --- | --- | --- |
| Import + QC | `checkpoints/qc_for_pca.mt` | yes |
| Global PCA | `global_pcs.tsv` | yes |

Set `PCA_SYNC_SCRIPTS=true` (default on Terra) to refresh `scripts/` from the
bucket on each restart. Force recomputation with:

- `PCA_FORCE_REIMPORT=true` — rebuild QC MatrixTable
- `PCA_FORCE_GLOBAL_PCA=true` — rerun global PCA (loads QC MT if needed)

If QC filter settings change, the notebook reruns import/QC even when a
checkpoint exists (params sidecar mismatch).

### Practical tips

- Use a **Hail** Cloud Environment (not a plain Python notebook).
- Stop or downsize the cluster after writing `global_pcs.tsv`.
- Keep `qc_for_pca.mt` for `tractor_05b` within-population PCA (05b joins
  covariates separately when ancestry labels are ready).

## Analysis design

- **Global PCs** are the primary association covariates (this notebook).
- Prefer an existing MatrixTable / VariantDataset if you already imported the
  joint DeepVariant callset. Otherwise fetch the Terra `GL_INTERVAL_set` data
  table via firecloud (columns **`VCF`** + optional **`VCF_idx`**; production
  joint-calling `chr*.g.vcf.bgz` shards). Keep autosomes only (`chr1`–`chr22`);
  drop `chrM` / `chrX` / `chrY`. Do **not** use FastFilter `output_vcf` unless
  you intentionally want that filtered callset. Every sample in the joint VCF
  is retained (including unreleased HG/NA controls), subject only to genotype
  call-rate filters.
- Filters: autosomes, biallelic SNVs, PASS-only variants, AF in `[0.01, 0.99]`,
  call-rate thresholds, and LD pruning before `hwe_normalized_pca`.

## Outputs

Under `gs://<workspace-bucket>/pca/<run_label>/`:

- `global_pcs.tsv` with `lr_PC1`–`lr_PC32` (parity with AoU short-read `PC1`–`PC32`)
- eigenvalues / loadings / run metadata
- `checkpoints/qc_for_pca.mt` (input to `tractor_05b`)


In [ ]:
from __future__ import annotations

from pathlib import Path
import os
import subprocess
import sys

# Bootstrap scripts/ from $WORKSPACE_BUCKET/scripts/ when not on the VM.
for _d in (Path.cwd() / "scripts", Path.cwd().parent / "scripts"):
    if (_d / "terra_notebook.py").is_file():
        sys.path.insert(0, str(_d.resolve()))
        break
else:
    _bucket = os.environ.get("WORKSPACE_BUCKET", "").rstrip("/")
    if not _bucket:
        raise FileNotFoundError(
            "scripts/ not found locally and WORKSPACE_BUCKET is unset. "
            "Upload scripts/ to gs://WORKSPACE/scripts/."
        )
    _dest = (Path.cwd() / "scripts").resolve()
    _dest.mkdir(parents=True, exist_ok=True)
    subprocess.check_call(
        ["gsutil", "-m", "rsync", "-r", f"{_bucket}/scripts/", str(_dest) + "/"]
    )
    sys.path.insert(0, str(_dest))

if os.environ.get("WORKSPACE_BUCKET", "").strip():
    os.environ.setdefault("PCA_SYNC_SCRIPTS", "true")

from terra_notebook import init_notebook

SCRIPTS = init_notebook(
    "workspace_paths.py",
    "resolve_gl_interval_manifest.py",
    "hail_pca_resume.py",
)
from workspace_paths import data_root
from resolve_gl_interval_manifest import (
    DEFAULT_ENTITY_TYPE as GL_INTERVAL_ENTITY_TYPE,
    DEFAULT_NAMESPACE as TERRA_NAMESPACE,
    DEFAULT_WORKSPACE as TERRA_WORKSPACE,
    fetch_gl_interval_manifest_firecloud,
    load_gl_interval_manifest_tsv,
    normalize_gl_interval_manifest,
)
from hail_pca_resume import (
    describe_stage,
    env_flag,
    hail_matrix_exists,
    output_exists,
    print_stage_plan,
    read_json_uri,
    write_json_uri,
)

import json

import pandas as pd

try:
    display
except NameError:
    def display(value):
        print(value)

ROOT = data_root()
WORKSPACE_BUCKET = os.environ.get("WORKSPACE_BUCKET", "").rstrip("/")
RUN_LABEL = os.environ.get("PCA_RUN_LABEL", "deepvariant_lr_v1")

if WORKSPACE_BUCKET:
    _bucket = (
        WORKSPACE_BUCKET
        if WORKSPACE_BUCKET.startswith("gs://")
        else f"gs://{WORKSPACE_BUCKET}"
    )
    CHROM_MANIFEST_URI = os.environ.get("PCA_CHROM_MANIFEST_URI", "")
    OUT_DIR = os.environ.get("PCA_OUT_DIR", f"{_bucket}/pca/{RUN_LABEL}")
    EXISTING_MT = os.environ.get("PCA_EXISTING_MT", f"{_bucket}/mt/deepvariant_joint.mt")
    EXISTING_VDS = os.environ.get("PCA_EXISTING_VDS", f"{_bucket}/vds/deepvariant_joint.vds")
else:
    CHROM_MANIFEST_URI = os.environ.get(
        "PCA_CHROM_MANIFEST_URI", str(ROOT / "manifests" / "GL_INTERVAL_set.tsv")
    )
    OUT_DIR = os.environ.get("PCA_OUT_DIR", str(ROOT / "pca" / RUN_LABEL))
    EXISTING_MT = os.environ.get("PCA_EXISTING_MT", "")
    EXISTING_VDS = os.environ.get("PCA_EXISTING_VDS", "")

INPUT_MODE = os.environ.get("PCA_INPUT_MODE", "vcf_list")
CHROM_ID_COLUMN = "entity:GL_INTERVAL_set_id"
CHROM_URI_COLUMN = "VCF"
CHROM_IDX_COLUMN = "VCF_idx"
VCF_URIS: list[str] | None = None
AUTOSOMES_ONLY = True

N_PCS = int(os.environ.get("PCA_N_PCS", "32"))
MIN_AF = 0.01
MAX_AF = 0.99
MIN_VARIANT_CALL_RATE = 0.98
MIN_SAMPLE_CALL_RATE = 0.98
LD_R2 = 0.1
LD_BP_WINDOW = 500_000
LD_MEMORY_PER_CORE = int(os.environ.get("PCA_LD_MEMORY_PER_CORE", "1"))
PASS_ONLY = True
CHROM_MANIFEST_SOURCE = os.environ.get(
    "PCA_CHROM_MANIFEST_SOURCE",
    "firecloud" if WORKSPACE_BUCKET and not CHROM_MANIFEST_URI else "tsv",
)
TERRA_ENTITY_TYPE = os.environ.get("PCA_TERRA_ENTITY_TYPE", GL_INTERVAL_ENTITY_TYPE)
RUN_PIPELINE = os.environ.get("PCA_RUN_PIPELINE", "").lower() in {"1", "true", "yes"}

CHECKPOINT_MT = f"{OUT_DIR}/checkpoints/qc_for_pca.mt"
QC_PARAMS_JSON = f"{OUT_DIR}/checkpoints/qc_for_pca.params.json"
GLOBAL_PCS_TSV = f"{OUT_DIR}/global_pcs.tsv"
GLOBAL_PCA_PARAMS_JSON = f"{OUT_DIR}/global_pca.params.json"
METADATA_JSON = f"{OUT_DIR}/run_metadata.global.json"

FORCE_REIMPORT = env_flag("PCA_FORCE_REIMPORT")
FORCE_GLOBAL_PCA = env_flag("PCA_FORCE_GLOBAL_PCA")
SYNC_SCRIPTS = env_flag("PCA_SYNC_SCRIPTS", default=bool(WORKSPACE_BUCKET))

QC_PARAMS = {
    "input_mode": INPUT_MODE,
    "pass_only": PASS_ONLY,
    "min_af": MIN_AF,
    "max_af": MAX_AF,
    "min_variant_call_rate": MIN_VARIANT_CALL_RATE,
    "min_sample_call_rate": MIN_SAMPLE_CALL_RATE,
}
GLOBAL_PCA_PARAMS = {
    "n_pcs": N_PCS,
    "ld_r2": LD_R2,
    "ld_bp_window": LD_BP_WINDOW,
    "ld_memory_per_core": LD_MEMORY_PER_CORE,
    "checkpoint_mt": CHECKPOINT_MT,
}

print("ROOT:", ROOT)
print("WORKSPACE_BUCKET:", WORKSPACE_BUCKET or "(local)")
print("OUT_DIR:", OUT_DIR)
print("INPUT_MODE:", INPUT_MODE)
print("CHROM_MANIFEST_SOURCE:", CHROM_MANIFEST_SOURCE)
print("SYNC_SCRIPTS:", SYNC_SCRIPTS)
print("FORCE_REIMPORT:", FORCE_REIMPORT)
print("FORCE_GLOBAL_PCA:", FORCE_GLOBAL_PCA)
print("RUN_PIPELINE:", RUN_PIPELINE)


In [ ]:
import hail as hl

# On Terra, use a Hail Cloud Environment with highmem workers (see markdown above).
# If LD prune / PCA OOMs, prefer fewer cores per executor so each gets more RAM:
#   hl.init(..., spark_conf={"spark.executor.cores": "4"})
if RUN_PIPELINE:
    hl.init(default_reference="GRCh38", idempotent=True)
    print("Hail version:", hl.version())
else:
    print("Dry run: Hail is not initialized until RUN_PIPELINE=True")


## 1. Resolve DeepVariant autosomal inputs

Read the Terra `GL_INTERVAL_set` data table via the firecloud API (default on
Terra), or fall back to a local TSV export / `PCA_CHROM_MANIFEST_URI`. Columns
used are `VCF` and optional `VCF_idx` from
`production_joint_calling/outputs/Chromosomes/`. Autosomes only. Hail discovers
the sibling `.tbi` beside each `VCF` path when `VCF_idx` is absent.


In [ ]:
def load_gl_interval_manifest(*, source: str) -> tuple[pd.DataFrame, str]:
    if source == "firecloud":
        manifest = fetch_gl_interval_manifest_firecloud(
            TERRA_NAMESPACE,
            TERRA_WORKSPACE,
            TERRA_ENTITY_TYPE,
            id_column=CHROM_ID_COLUMN,
        )
        source_label = f"{TERRA_NAMESPACE}/{TERRA_WORKSPACE}/{TERRA_ENTITY_TYPE}"
    else:
        manifest_uri = CHROM_MANIFEST_URI
        if not manifest_uri:
            candidates = [
                ROOT / "manifests" / "GL_INTERVAL_set.tsv",
                ROOT / "tractor_mix" / "manifests" / "GL_INTERVAL_set.tsv",
                Path.cwd() / "tractor_mix" / "manifests" / "GL_INTERVAL_set.tsv",
            ]
            for local_manifest in candidates:
                if local_manifest.exists():
                    manifest_uri = str(local_manifest)
                    break
            else:
                raise FileNotFoundError(
                    "No GL_INTERVAL_set source found. Set PCA_CHROM_MANIFEST_SOURCE=firecloud "
                    "on Terra, or export the table to a local TSV / set PCA_CHROM_MANIFEST_URI."
                )
        if manifest_uri.startswith("gs://") or Path(manifest_uri).exists():
            manifest = pd.read_csv(manifest_uri, sep="\t", dtype=str)
        else:
            manifest = load_gl_interval_manifest_tsv(Path(manifest_uri), id_column=CHROM_ID_COLUMN)
        source_label = manifest_uri

    out = normalize_gl_interval_manifest(
        manifest,
        id_column=CHROM_ID_COLUMN,
        uri_column=CHROM_URI_COLUMN,
        idx_column=CHROM_IDX_COLUMN,
        autosomes_only=AUTOSOMES_ONLY,
        source_label=source_label,
    )
    return out, source_label


def natural_chrom_key(interval_id: str) -> tuple[int, str]:
    token = str(interval_id).lower()
    for chrom in range(1, 23):
        if token == f"chr{chrom}":
            return chrom, token
    return 999, token


if VCF_URIS is not None:
    chrom_manifest = pd.DataFrame({
        "interval_id": [f"uri{i}" for i in range(len(VCF_URIS))],
        CHROM_URI_COLUMN: list(VCF_URIS),
    })
    manifest_source = "VCF_URIS"
    vcf_uris = list(VCF_URIS)
elif INPUT_MODE == "vcf_list":
    manifest_source = CHROM_MANIFEST_SOURCE
    if not RUN_PIPELINE and manifest_source == "firecloud":
        manifest_source = "tsv"
    chrom_manifest, manifest_source = load_gl_interval_manifest(source=manifest_source)
    chrom_manifest = chrom_manifest.sort_values(
        "interval_id", key=lambda s: s.map(natural_chrom_key)
    )
    vcf_uris = chrom_manifest[CHROM_URI_COLUMN].tolist()
    print(f"Manifest source: {manifest_source}")
    print(f"Intervals: {', '.join(chrom_manifest['interval_id'])}")
    if CHROM_IDX_COLUMN in chrom_manifest.columns:
        mismatched = chrom_manifest.loc[
            ~chrom_manifest[CHROM_IDX_COLUMN].str.endswith(".tbi")
            | ~chrom_manifest.apply(
                lambda r: r[CHROM_IDX_COLUMN].startswith(r[CHROM_URI_COLUMN]),
                axis=1,
            )
        ]
        if len(mismatched):
            print(f"WARNING: {len(mismatched)} VCF/VCF_idx pairs look mismatched")
            display(mismatched[["interval_id", CHROM_URI_COLUMN, CHROM_IDX_COLUMN]])
else:
    chrom_manifest = pd.DataFrame()
    manifest_source = ""
    vcf_uris = []

print(f"Resolved {len(vcf_uris)} autosomal shards")
for uri in vcf_uris[:5]:
    print(" ", uri)
if len(vcf_uris) > 5:
    print("  ...")


## 2. Import / checkpoint a QC MatrixTable for PCA

If contig names are already `chr1`…`chr22`, leave `FIND_REPLACE=None`.
If they are bare `1`…`22`, set `FIND_REPLACE` appropriately or rely on
Hail's contig recoding for GRCh38.


In [ ]:
FIND_REPLACE = None  # e.g. ("^([0-9]+)$", "chr\\1") only if needed


def import_genotype_mt():
    if INPUT_MODE == "mt":
        return hl.read_matrix_table(EXISTING_MT)
    if INPUT_MODE == "vds":
        vds = hl.vds.read_vds(EXISTING_VDS)
        return hl.vds.to_dense_mt(vds)
    if INPUT_MODE == "vcf_list":
        assert vcf_uris, "Set VCF_URIS or CHROM_MANIFEST_URI for INPUT_MODE='vcf_list'"
        kwargs = dict(
            path=vcf_uris,
            force_bgz=True,
            reference_genome="GRCh38",
            array_elements_required=False,
        )
        if FIND_REPLACE is not None:
            kwargs["find_replace"] = FIND_REPLACE
        return hl.import_vcf(**kwargs)
    raise ValueError(f"Unsupported INPUT_MODE={INPUT_MODE!r}")


def filter_for_pca(mt):
    mt = mt.filter_rows(mt.locus.in_autosome())
    mt = mt.filter_rows(hl.len(mt.alleles) == 2)
    mt = mt.filter_rows(hl.is_snp(mt.alleles[0], mt.alleles[1]))
    if PASS_ONLY:
        mt = mt.filter_rows(hl.is_missing(mt.filters) | (hl.len(mt.filters) == 0))
    mt = hl.variant_qc(mt)
    mt = hl.sample_qc(mt)
    mt = mt.filter_rows(
        (mt.variant_qc.AF[1] >= MIN_AF)
        & (mt.variant_qc.AF[1] <= MAX_AF)
        & (mt.variant_qc.call_rate >= MIN_VARIANT_CALL_RATE)
    )
    mt = mt.filter_cols(mt.sample_qc.call_rate >= MIN_SAMPLE_CALL_RATE)
    return mt.checkpoint(CHECKPOINT_MT, overwrite=True)


def load_qc_matrixtable() -> hl.MatrixTable:
    return hl.read_matrix_table(CHECKPOINT_MT)


qc_stage = describe_stage(
    "import + QC MatrixTable",
    CHECKPOINT_MT,
    force=FORCE_REIMPORT,
    params_path=QC_PARAMS_JSON,
    current_params=QC_PARAMS,
    matrix=True,
)

global_stage = describe_stage(
    "global PCA",
    GLOBAL_PCS_TSV,
    force=FORCE_GLOBAL_PCA,
    params_path=GLOBAL_PCA_PARAMS_JSON,
    current_params=GLOBAL_PCA_PARAMS,
)
print_stage_plan([qc_stage, global_stage])

if RUN_PIPELINE:
    if qc_stage["action"] == "skip":
        print(f"RESUME: loading QC MatrixTable from {CHECKPOINT_MT}")
        mt = load_qc_matrixtable()
        n_variants, n_samples = mt.count()
        print(f"PCA QC MT: {n_variants:,} variants x {n_samples:,} samples")
    else:
        if qc_stage["exists"] and qc_stage["action"].startswith("rerun"):
            print("Params changed or force flag set; rebuilding QC MatrixTable")
        mt = import_genotype_mt()
        print("Imported MT:", mt.count())
        mt = filter_for_pca(mt)
        write_json_uri(QC_PARAMS_JSON, QC_PARAMS)
        n_variants, n_samples = mt.count()
        print(f"PCA QC MT: {n_variants:,} variants x {n_samples:,} samples")
        print("wrote", CHECKPOINT_MT)
else:
    print("Dry run: genotype import / QC skipped")


## 3. Global LD-pruned PCA

This is the primary ancestry covariate set for association models.


In [ ]:
def run_pca(mt, *, n_pcs: int):
    pruned = hl.ld_prune(
        mt.GT,
        r2=LD_R2,
        bp_window_size=LD_BP_WINDOW,
        memory_per_core=LD_MEMORY_PER_CORE,
    )
    mt_pruned = mt.filter_rows(hl.is_defined(pruned[mt.row_key]))
    eigenvalues, scores, loadings = hl.hwe_normalized_pca(
        mt_pruned.GT,
        k=n_pcs,
        compute_loadings=True,
    )
    return scores, loadings, eigenvalues


def scores_to_dataframe(scores, *, prefix: str) -> pd.DataFrame:
    pdf = scores.to_pandas()
    pc_cols = [f"{prefix}{i}" for i in range(1, N_PCS + 1)]
    expanded = pd.DataFrame(pdf["scores"].tolist(), columns=pc_cols)
    out = pd.concat([pdf[["s"]].rename(columns={"s": "research_id"}), expanded], axis=1)
    assert out["research_id"].is_unique
    return out


if RUN_PIPELINE:
    if global_stage["action"] == "skip":
        print(f"RESUME: using existing global PCs at {GLOBAL_PCS_TSV}")
        global_pcs = pd.read_csv(GLOBAL_PCS_TSV, sep="\t", dtype={"research_id": str})
        display(global_pcs.head())
    else:
        if "mt" not in globals():
            if hail_matrix_exists(CHECKPOINT_MT):
                print(f"Loading QC MatrixTable from {CHECKPOINT_MT}")
                mt = load_qc_matrixtable()
            else:
                raise RuntimeError(
                    "QC MatrixTable missing. Run the import/QC cell first, "
                    f"or set PCA_FORCE_REIMPORT=true. Expected: {CHECKPOINT_MT}"
                )
        global_scores, global_loadings, global_eigenvalues = run_pca(mt, n_pcs=N_PCS)
        global_pcs = scores_to_dataframe(global_scores, prefix="lr_PC")
        hl.Table.from_pandas(global_pcs).export(GLOBAL_PCS_TSV)
        global_loadings.write(f"{OUT_DIR}/global_loadings.ht", overwrite=True)
        with hl.hadoop_open(f"{OUT_DIR}/global_eigenvalues.txt", "w") as handle:
            for value in global_eigenvalues:
                handle.write(f"{value}\n")
        write_json_uri(GLOBAL_PCA_PARAMS_JSON, GLOBAL_PCA_PARAMS)
        display(global_pcs.head())
        print("wrote", GLOBAL_PCS_TSV)
else:
    print("Dry run: global PCA skipped")


## 4. Provenance and next steps

Review PC plots and association inflation before replacing short-read PCs.
Recommended integration later: add `lr_PC1`–`lr_PC32` beside the existing AoU
short-read PCs rather than overwriting them.

**Within-population PCs:** after covariates have filled ancestry labels,
run `tractor_05b_pca_within_population.ipynb` against the same `RUN_LABEL` /
`qc_for_pca.mt` checkpoint (05b loads covariates itself).


In [ ]:
from datetime import datetime, timezone

metadata = {
    "run_label": RUN_LABEL,
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "engine": "hail",
    "pca_scope": "global",
    "input_mode": INPUT_MODE,
    "paths": {
        "workspace_bucket": WORKSPACE_BUCKET,
        "existing_mt": EXISTING_MT,
        "existing_vds": EXISTING_VDS,
        "chrom_manifest_source": CHROM_MANIFEST_SOURCE,
        "chrom_manifest": CHROM_MANIFEST_URI or None,
        "terra_namespace": TERRA_NAMESPACE,
        "terra_workspace": TERRA_WORKSPACE,
        "terra_entity_type": TERRA_ENTITY_TYPE,
        "chrom_id_column": CHROM_ID_COLUMN,
        "chrom_uri_column": CHROM_URI_COLUMN,
        "chrom_idx_column": CHROM_IDX_COLUMN,
        "autosomes_only": AUTOSOMES_ONLY,
        "out_dir": OUT_DIR,
        "checkpoint_mt": CHECKPOINT_MT,
        "global_pcs": GLOBAL_PCS_TSV,
    },
    "cohort": {
        "definition": "all samples in joint VCF (genotype call-rate filter only)",
    },
    "filters": {
        "autosomal_biallelic_snps": True,
        "pass_only": PASS_ONLY,
        "min_af": MIN_AF,
        "max_af": MAX_AF,
        "min_variant_call_rate": MIN_VARIANT_CALL_RATE,
        "min_sample_call_rate": MIN_SAMPLE_CALL_RATE,
        "ld_r2": LD_R2,
        "ld_bp_window": LD_BP_WINDOW,
        "n_pcs": N_PCS,
    },
    "resume": {
        "qc_stage": qc_stage if RUN_PIPELINE else None,
        "global_stage": global_stage if RUN_PIPELINE else None,
        "force_reimport": FORCE_REIMPORT,
        "force_global_pca": FORCE_GLOBAL_PCA,
    },
    "status": "completed" if RUN_PIPELINE else "dry_run",
    "next_step": "tractor_05b_pca_within_population.ipynb",
}

if RUN_PIPELINE:
    with hl.hadoop_open(METADATA_JSON, "w") as handle:
        handle.write(json.dumps(metadata, indent=2) + "\n")
else:
    local_out = ROOT / "pca" / RUN_LABEL
    local_out.mkdir(parents=True, exist_ok=True)
    (local_out / "run_metadata.global.dry_run.json").write_text(
        json.dumps(metadata, indent=2) + "\n"
    )
    print("wrote", local_out / "run_metadata.global.dry_run.json")

print(json.dumps(metadata, indent=2))
